RUNTIME: T4 or A100     
INSTRUCTIONS:

1. Set runtime to GPU: Runtime → Change runtime type → GPU → T4 or A100
2. Run all cells in order: Runtime → Run all
3. When prompted, authorize Google Drive access
4. Model checkpoints are saved to Google Drive via:
```
python   model.save_pretrained(SAVE_DIR + "checkpoints/setfit_model")
```
Make sure SAVE_DIR is set correctly in the setup cell   

5. To save results/figures to GitHub:   
    a. Right-click the file in the left sidebar → Download  
    b. Go to the GitHub repo → results/ → Add file → Upload files   
    c. Commit directly on GitHub    
6. Save the notebook: Ctrl+S → save to GitHub when prompted

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────
!pip install -q setfit accelerate

# ── 2. Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"

# ── 3. Standard imports ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from setfit import SetFitModel, Trainer, TrainingArguments

# ── 4. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Shared Evaluation Wiring

This section integrates `src/evaluate.py` so SetFit uses the same metrics/report format as other models.
Run this cell once, then call `evaluate_setfit_predictions(...)` after generating `y_pred_binary` for test data.


In [ ]:
import ast
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer

# Resolve repo root so src/ imports work in both local Jupyter and Colab
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src" / "evaluate.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo root containing src/evaluate.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_run, save_evaluation_outputs

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()

    def _parse_one(value):
        if isinstance(value, list):
            return [int(x) for x in value]
        if isinstance(value, np.ndarray):
            return [int(x) for x in value.tolist()]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if body == "":
                    return []
                if ',' in body:
                    tokens = [t.strip() for t in body.split(',') if t.strip()]
                else:
                    tokens = [t for t in body.split() if t]
                return [int(t) for t in tokens]
            return [int(x) for x in ast.literal_eval(s)]
        return value

    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _resolve_data_dir():
    # Local notebooks usually run from notebooks/, so data is often ../data
    for candidate in [repo_root / "data", Path("../data"), Path("data")]:
        if (candidate / "test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory containing test.csv")

def _get_label_names():
    dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
    return dataset["train"].features["labels"].feature.names

def evaluate_setfit_predictions(
    y_pred_binary,
    *,
    data_fraction,
    seed,
    method="setfit",
    data_dir=None,
    output_dir=None,
):
    """
    Evaluate SetFit predictions on the shared test split and save standard outputs.

    y_pred_binary: binary matrix (n_samples, n_labels) aligned with test.csv rows.
    """
    data_dir = Path(data_dir) if data_dir else _resolve_data_dir()
    output_dir = Path(output_dir) if output_dir else (repo_root / "results")

    test_df = pd.read_csv(data_dir / "test.csv")
    test_df = _parse_labels_column(test_df, label_col="labels")

    label_names = _get_label_names()
    mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
    mlb.fit([list(range(len(label_names)))])
    y_true_binary = mlb.transform(test_df["labels"])

    evaluation = evaluate_run(
        method=method,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_true_binary,
        y_pred=y_pred_binary,
    )

    paths = save_evaluation_outputs(evaluation, method=method, output_dir=output_dir)
    return evaluation, paths

print("Shared evaluation helpers loaded.")
print("After generating SetFit predictions, call evaluate_setfit_predictions(y_pred_binary, data_fraction=..., seed=...)")


## 3. SetFit Experiment Config

Configure fractions, seeds, model, and output paths for SetFit runs.


In [ ]:
# --- SetFit experiment config ---
METHOD_NAME = "setfit"
MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"

# Include full-data run (1.00) for low-data vs full-data comparisons
DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 7, 21]

TRAIN_FILE_BY_FRACTION = {
    0.01: "train_1pct.csv",
    0.05: "train_5pct.csv",
    0.10: "train_10pct.csv",
    0.25: "train_25pct.csv",
    0.50: "train_50pct.csv",
    1.00: "train.csv",
}

SETFIT_CONFIG = {
    "num_iterations": 20,
    "num_epochs": 1,
    "batch_size": 16,
    "body_learning_rate": 2e-5,
    "head_learning_rate": 1e-2,
}

RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Optional checkpoint save for full-data runs (if SAVE_DIR exists from setup cell)
ENABLE_CHECKPOINT_SAVE = "SAVE_DIR" in globals()
if ENABLE_CHECKPOINT_SAVE:
    CHECKPOINT_ROOT = Path(SAVE_DIR) / "checkpoints" / "setfit"
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Config loaded.")
print("Results dir:", RESULTS_DIR)


## 4. Run SetFit Experiments

Trains SetFit across configured fractions and seeds, evaluates with shared framework, and writes result CSVs.


In [ ]:
import random
from datasets import Dataset

def _set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

def _choose_text_col(df):
    for col in ["text_clean_transformer", "text"]:
        if col in df.columns:
            return col
    raise ValueError("No text column found. Expected one of: text_clean_transformer, text")

def _to_binary_matrix(label_lists, n_labels):
    mlb = MultiLabelBinarizer(classes=list(range(n_labels)))
    mlb.fit([list(range(n_labels))])
    return mlb.transform(label_lists)

def _predictions_to_binary(preds, n_labels):
    # Case 1: already matrix-like
    try:
        arr = np.asarray(preds)
        if arr.ndim == 2 and arr.shape[1] == n_labels:
            return (arr > 0).astype(int)
    except Exception:
        pass

    # Case 2: sequence of predicted label-id collections
    out = np.zeros((len(preds), n_labels), dtype=int)
    for i, pred in enumerate(preds):
        if isinstance(pred, (list, tuple, set, np.ndarray)):
            for label_id in pred:
                try:
                    idx = int(label_id)
                    if 0 <= idx < n_labels:
                        out[i, idx] = 1
                except Exception:
                    continue
        else:
            try:
                idx = int(pred)
                if 0 <= idx < n_labels:
                    out[i, idx] = 1
            except Exception:
                continue
    return out

def _build_setfit_model(model_name):
    # Prefer explicit multilabel strategy when supported
    try:
        return SetFitModel.from_pretrained(model_name, multi_target_strategy="one-vs-rest")
    except TypeError:
        return SetFitModel.from_pretrained(model_name)

def _build_trainer(model, train_dataset, eval_dataset, cfg):
    last_exc = None

    # Newer API path
    try:
        args = TrainingArguments(
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            num_iterations=cfg["num_iterations"],
            body_learning_rate=cfg["body_learning_rate"],
            head_learning_rate=cfg["head_learning_rate"],
        )
        return Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            column_mapping={"text": "text", "label": "label"},
        )
    except Exception as exc:
        last_exc = exc

    # Legacy API fallback
    try:
        return Trainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            num_iterations=cfg["num_iterations"],
            column_mapping={"text": "text", "label": "label"},
        )
    except Exception as exc:
        last_exc = exc

    raise RuntimeError(f"Could not construct SetFit Trainer with known APIs: {last_exc}")

def run_single_setfit_experiment(data_fraction, seed):
    data_dir = _resolve_data_dir()

    train_filename = TRAIN_FILE_BY_FRACTION[float(data_fraction)]
    train_path = data_dir / train_filename
    test_path = data_dir / "test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing train file for fraction={data_fraction}: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test file: {test_path}")

    train_df = _parse_labels_column(pd.read_csv(train_path), label_col="labels")
    test_df = _parse_labels_column(pd.read_csv(test_path), label_col="labels")

    text_col = _choose_text_col(train_df)
    label_names = _get_label_names()
    n_labels = len(label_names)

    y_train = _to_binary_matrix(train_df["labels"], n_labels)
    y_test = _to_binary_matrix(test_df["labels"], n_labels)

    train_dataset = Dataset.from_dict({
        "text": train_df[text_col].fillna("").astype(str).tolist(),
        "label": y_train.tolist(),
    })
    eval_dataset = Dataset.from_dict({
        "text": test_df[text_col].fillna("").astype(str).tolist(),
        "label": y_test.tolist(),
    })

    _set_global_seed(int(seed))
    model = _build_setfit_model(MODEL_NAME)
    trainer = _build_trainer(model, train_dataset, eval_dataset, SETFIT_CONFIG)
    trainer.train()

    raw_preds = model.predict(test_df[text_col].fillna("").astype(str).tolist())
    y_pred = _predictions_to_binary(raw_preds, n_labels)

    evaluation = evaluate_run(
        method=METHOD_NAME,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_test,
        y_pred=y_pred,
    )

    if ENABLE_CHECKPOINT_SAVE and float(data_fraction) == 1.00:
        ckpt_dir = CHECKPOINT_ROOT / f"seed_{seed}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(ckpt_dir))

    return evaluation

all_overall = []
all_per_class = []
run_errors = []

for seed in SEEDS:
    for frac in DATA_FRACTIONS:
        print(f"\n=== SetFit run: fraction={frac}, seed={seed} ===")
        try:
            eval_out = run_single_setfit_experiment(frac, seed)
            all_overall.append(eval_out["overall"])
            all_per_class.append(eval_out["per_class"])

            macro_f1 = eval_out["overall"].iloc[0]["macro_f1"]
            print(f"Completed: macro_f1={macro_f1:.4f}")
        except Exception as exc:
            run_errors.append({"seed": seed, "data_fraction": frac, "error": str(exc)})
            print(f"FAILED: {exc}")

if all_overall:
    overall_df = pd.concat(all_overall, ignore_index=True).sort_values(["seed", "data_fraction"])
    per_class_df = pd.concat(all_per_class, ignore_index=True).sort_values(["seed", "data_fraction", "emotion"])

    overall_path = RESULTS_DIR / "setfit_overall.csv"
    per_class_path = RESULTS_DIR / "setfit_per_class.csv"
    readme_path = RESULTS_DIR / "setfit_results.csv"

    overall_df.to_csv(overall_path, index=False)
    per_class_df.to_csv(per_class_path, index=False)

    # README-compatible long format
    readme_df = per_class_df[["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
    readme_df.to_csv(readme_path, index=False)

    print("\nSaved outputs:")
    print(overall_path)
    print(per_class_path)
    print(readme_path)
else:
    print("No successful runs; no result files were saved.")

if run_errors:
    errors_df = pd.DataFrame(run_errors)
    print("\nRun errors:")
    display(errors_df)
else:
    print("\nAll runs completed without exceptions.")
